### ハミング距離の計算

### CWALDP

### RAPPOR

In [14]:
import numpy as np
import json
import math

def make_seed(sample_id, l, noise_p, salt):
    import hashlib
    h = hashlib.blake2b(digest_size=8)
    h.update(f"{sample_id}-{l}-{noise_p}-{salt}".encode())
    return int.from_bytes(h.digest(), 'big', signed=False)

def add_flip_noise_packed(packed_bf, l, noise_p, sample_id, salt):
    if noise_p <= 0.0:
        return packed_bf
    rng = np.random.default_rng(make_seed(sample_id, l, noise_p, salt))
    flips_bits = (rng.random(l) < noise_p).astype(np.uint8)
    flips_packed = np.packbits(flips_bits)
    return np.bitwise_xor(packed_bf, flips_packed)


HASH_NUMBER=1
seed=1
for eps in [0.5]:
    
    noise_p = 1 / (1 + math.exp(eps / (2 * HASH_NUMBER)))

    # --- 1) データ読み込み 
    original_path="../../data/FashionMNIST/BF/fmnist_zw_fp0.4_n3_bf_length2617_k1_PI0.5_L16.npz"
    dat = np.load(original_path, allow_pickle=True)
    print(dat)
    bf= dat["X_bits"].astype(np.uint8)#パックされたデータ
    meta = json.loads(dat["meta_json"].item())
    print(meta)
    l = meta["bf_length"]
    print("BF_length",l)

    X_noisy_packed = [
        add_flip_noise_packed(bf[i], l, noise_p, i, seed)
        for i in range(len(bf))
    ]
    print(np.bitwise_xor(bf,X_noisy_packed)[:5])
    hamming_weight=np.bitwise_xor(bf,X_noisy_packed).sum(axis=1)
    print(bf.shape)
    print(hamming_weight.shape)
   
    #print(np.mean(hamming_weight),np.var(hamming_weight))

NpzFile '../../data/FashionMNIST/BF/fmnist_zw_fp0.4_n3_bf_length2617_k1_PI0.5_L16.npz' with keys: X_bits, y, meta_json
{'fp': 0.4, 'neighbors': 3, 'noise_p': 0.0, 'bf_length': 2617, 'hash_number': 1}
BF_length 2617
[[180 145 181 ...  92 198 128]
 [ 36 145  74 ...  18 227 128]
 [ 21  73   8 ... 200 164   0]
 [139  49 164 ... 122  74 128]
 [131  38  89 ...  19  55   0]]
(70000, 328)
(70000,)


In [21]:
print(bf.shape)
print(len(X_noisy_packed))

(70000, 328)
70000
